# time-stage-instrumentation — worked example 2: Report sync as a fraction of step time

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `time-stage-instrumentation`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A common DDP diagnostic is 'what fraction of each step is gradient sync?'. You time the whole step and the sync sub-stage separately, then divide. If the fraction is high, communication — not compute — is your bottleneck.

## Worked solution

We profile a step that has a compute part and an all-reduce (sync) part, and report the sync share.

1. We open a `step` timer with `perf_counter` at the very top of the iteration body.
2. Inside, we separately time only the sync sub-stage and accumulate it.
3. We close the `step` timer at the end of the iteration and accumulate the full step time.
4. After the loop, `sync_total / step_total` is the fraction of wall time spent synchronizing.

Since the sync sleep is half the compute sleep, the printed fraction sits sensibly between 0 and 1 — a number you would log to wandb to decide whether to overlap communication with compute.

In [ ]:
import time

def sync_fraction(n_iters, sleep_compute, sleep_sync):
    step_total = 0.0
    sync_total = 0.0
    for _ in range(n_iters):
        step_t0 = time.perf_counter()
        time.sleep(sleep_compute)
        sync_t0 = time.perf_counter()
        time.sleep(sleep_sync)
        sync_total += time.perf_counter() - sync_t0
        step_total += time.perf_counter() - step_t0
    return sync_total / step_total

frac = sync_fraction(3, 0.01, 0.005)
print('sync fraction:', round(frac, 3))
print('in range:', 0.0 < frac < 1.0)